In [81]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [82]:
seed = 42
df = sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [83]:
def ols(X, y):
  return np.linalg.inv(X.T @ X) @ X.T @ y


def gradient_descent(X, y, learning_rate=0.01, n_iterations=1000, alpha=0.01):
  y = y.reshape(-1, 1)
  n = y.shape[0]
  p = X.shape[1]
  B = np.zeros(p).reshape(-1, 1)
  for i in range(n_iterations):
    gradient = 1/n * X.T @ (X@B - y) + alpha * np.sign(B)
    B -= learning_rate * gradient
  return B.reshape(-1)

def mse(y, pred):
  return np.mean((y - pred) ** 2)

def rmse(y, pred):
  return np.sqrt(mse(y, pred))

def mae(y, pred):
  return np.mean(np.abs(y - pred))

def r_squared(y, pred):
  mean_y = np.mean(y)
  return 1 - (np.sum((y - pred) ** 2) / np.sum((y - mean_y) ** 2))

def adjusted_r_squared(y, pred, n_features):
  return 1 - (
    ((1 - r_squared(y, pred)) * (y.shape[0] - 1)) / 
    (y.shape[0] - n_features - 1)
  )


In [84]:


#! Later i might generate new feautres (feature engineering)
# df['tip_percentage'] = df['tip'] / df['total_bill'] * 100
# df['bill_per_person'] = df['total_bill'] / df['size']


In [85]:
df['sex'] = df['sex'].cat.codes
df['smoker'] = df['smoker'].cat.codes
df['time'] = df['time'].cat.codes

In [108]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def produce_training_data(df, features):
  X = df[features]
  y = df['tip']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

  scaler = StandardScaler()
  X_train_standard = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  assert X_train_standard.shape == X_train.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_test_scaled.shape == X_test.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_train.shape[0] == y_train.shape[0], "X train's shape does not match the y train's shape"
  assert X_test.shape[0] == y_test.shape[0], "X test's shape does not match the y test's shape"
  assert X_train.shape[1] == X.shape[1] == X_test.shape[1], "Features shape does not match in X_train, X_test, and original X"


  X_train_standard = np.concat((np.ones(X_train_standard.shape[0]).reshape(-1, 1), X_train_standard), axis=1)
  X_test_scaled = np.concat((np.ones(X_test_scaled.shape[0]).reshape(-1, 1), X_test_scaled), axis=1)

  return X_train_standard, X_test_scaled, y_train.to_numpy(), y_test.to_numpy()


In [ ]:
#! Checking to see if log scale for total_bill works better or not

# df['log_total_bill'] = np.log(df['total_bill'])
# features = ['log_total_bill']
# X_train, X_test, y_train, y_test = produce_training_data(df, features)

# res_ols = ols(X_train, y_train)
# pred_ols = X_test @ res_ols

# pd.DataFrame({
#   "MSE": mse(y_test, pred_ols),
#   "RMSE": rmse(y_test, pred_ols), 
#   "MAE": mae(y_test, pred_ols), 
#   "R2": r_squared(y_test, pred_ols), 
#   "Adjusted_R2": adjusted_r_squared(y_test, pred_ols, X_test.shape[1]),
#   "Features": " | ".join(features),
#   "Algorithm": "OLS"
# })

,MSE,RMSE,MAE,R2,Adjusted_R2,Features,Algorithm
0,0.607329,0.779313,0.602059,0.514126,0.493001,log_total_bill,OLS


In [ ]:
# Feature selection with Lasso regression

features = ['total_bill', 'size', 'sex', 'smoker', 'time']
X_train, X_test, y_train, y_test = produce_training_data(df, features)

weight_names = ['B' , *features]

,B,total_bill,size,sex,smoker,time,alpha
0,3.087695,0.821680,0.224872,-0.015176,0.091554,-0.023269,0.0001
1,3.086795,0.820696,0.224547,-0.014184,0.090632,-0.021963,0.0010
2,3.077795,0.810849,0.221289,-0.004261,0.081419,-0.008903,0.0100
3,2.987795,0.731330,0.184583,-0.000825,0.000192,0.000403,0.1000
4,2.087795,0.003964,0.001975,0.005068,0.002422,0.002106,1.0000
5,0.085745,-0.072024,0.007762,-0.100440,-0.020247,-0.041067,10.0000
6,0.029791,-0.454708,-0.029395,0.409603,-0.487519,-0.366158,100.0000


In [162]:


def find_ideal_alpha(alphas):
  result = []
  for alpha in alphas:
    weights = gradient_descent(X_train, y_train, n_iterations=10000, alpha=alpha)
    pred = X_test @ weights
    result.append([
      alpha,
      mse(y_test, pred)
    ])

  result = np.array(result)
  ideal_alpha = result[result[:, 1].argmin()][0]
  return ideal_alpha

In [165]:
ideal_alpha = find_ideal_alpha([0.0001 * 10 ** p for p in range(7)])
ideal_alpha
# Lowest MSE for alpha = 0.1
# 0.01 < ideal alpha < 1

np.float64(0.1)

In [171]:
ideal_alpha = find_ideal_alpha([*[1 - 0.1 * p for p in range(10)], *[0.1 - 0.01 * p for p in range(10)]])
ideal_alpha = round(ideal_alpha, 1)
ideal_alpha

# ideal alpha = 0.2
# 0.1 < ideal alpha < 0.3

np.float64(0.2)

In [ ]:
ideal_alpha = find_ideal_alpha([0.3 - 0.01 * p for p in range(20)])
ideal_alpha

# ideal alpha = 0.24

np.float64(0.24)

In [175]:
weights = gradient_descent(X_train, y_train, n_iterations=10000, alpha=ideal_alpha)
weights

array([ 2.84779487e+00,  6.41834020e-01,  9.51547739e-02,  1.54295970e-03,
       -9.82081856e-04,  2.31223649e-03])

In [173]:
features = []
results = []

for feature in ['total_bill', 'size', 'sex', 'smoker', 'time']:
  features.append(feature)

  X_train, X_test, y_train, y_test = produce_training_data(df, features)

  res_ols = ols(X_train, y_train)
  pred_ols = X_test @ res_ols

  results.append({
    "MSE": mse(y_test, pred_ols),
    "RMSE": rmse(y_test, pred_ols), 
    "MAE": mae(y_test, pred_ols), 
    "R2": r_squared(y_test, pred_ols), 
    "Adjusted_R2": adjusted_r_squared(y_test, pred_ols, X_test.shape[1]),
    "Features": " | ".join(features),
    "Algorithm": "OLS"
  })


  res_lasso = gradient_descent(X_train, y_train, n_iterations=10000, alpha=ideal_alpha)
  pred_lasso = X_test @ res_lasso
  results.append({
      "MSE": mse(y_test, pred_lasso),
      "RMSE": rmse(y_test, pred_lasso), 
      "MAE": mae(y_test, pred_lasso), 
      "R2": r_squared(y_test, pred_lasso), 
      "Adjusted_R2": adjusted_r_squared(y_test, pred_lasso, X_test.shape[1]),
      "Features": " | ".join(features),
      "Algorithm": "Lasso"
    })


pd.DataFrame(results)

,MSE,RMSE,MAE,R2,Adjusted_R2,Features,Algorithm
0,0.568814,0.754198,0.620858,0.544938,0.525153,total_bill,OLS
1,0.525113,0.724647,0.589563,0.579900,0.561635,total_bill,Lasso
2,0.648600,0.805357,0.663924,0.481108,0.446516,total_bill | size,OLS
3,0.535273,0.731623,0.599868,0.571772,0.543223,total_bill | size,Lasso
4,0.650938,0.806807,0.664623,0.479238,0.431896,total_bill | size | sex,OLS
5,0.535255,0.731611,0.599878,0.571786,0.532858,total_bill | size | sex,Lasso
6,0.689392,0.830296,0.668122,0.448474,0.384343,total_bill | size | sex | smoker,OLS
7,0.536647,0.732562,0.600189,0.570672,0.520751,total_bill | size | sex | smoker,Lasso
8,0.694797,0.833545,0.670306,0.444150,0.364742,total_bill | size | sex | smoker | time,OLS
9,0.534422,0.731042,0.599707,0.572452,0.511374,total_bill | size | sex | smoker | time,Lasso


In [89]:
# residual = (y_test - pred).to_numpy()

# sns.kdeplot(x=residual)
